# 🧠 DatasheetIQ — Knowledge Graph Backend (Google Colab)

Runs **FastAPI backend + Qwen 3.5 4B** in Google Colab.
The model is downloaded during the installation phase.

In [ ]:
# CELL 1: Clean and uninstall existing (to avoid version conflicts with Colab default)
!pip uninstall -y fastapi starlette httpx uvicorn pydantic python-multipart 2>/dev/null
print("✅ Environment cleaned")

In [ ]:
# CELL 2: Install Libraries + Qwen 3.5 4B Model weights
!pip install fastapi==0.124.1 uvicorn==0.34.0 starlette==0.49.1 pydantic==2.12.0 pydantic-settings httpx==0.28.1 python-multipart==0.0.18 neo4j==5.23.0 pyngrok PyMuPDF==1.24.0 pdfplumber==0.11.0 transformers==4.45.0 accelerate==0.34.0 torch

print("\n📦 Downloading Qwen 3.5 4B Model (this may take 2-4 minutes)...")
!python -c "from transformers import AutoModelForCausalLM, AutoTokenizer; mid='Qwen/Qwen3.5-4B'; AutoTokenizer.from_pretrained(mid, trust_remote_code=True); AutoModelForCausalLM.from_pretrained(mid, trust_remote_code=True)"

print("\n✅ ALL INSTALLED: Libraries + Qwen 3.5 4B weights ready")

In [ ]:
# CELL 3: Mount Google Drive & Set Project Root
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Working in {os.getcwd()}")

In [ ]:
# CELL 4: Validate components
import fastapi, uvicorn, pydantic, transformers, torch
print(f"fastapi: {fastapi.__version__}")
print(f"GPU: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0)}")

In [ ]:
# CELL 5: Set Credentials
import os
os.environ['NEO4J_URI'] = 'neo4j+s://xxxxxxxx.databases.neo4j.io'  # ← REPLACE
os.environ['NEO4J_USER'] = 'neo4j'                                  # ← REPLACE
os.environ['NEO4J_PASSWORD'] = 'your-password-here'                  # ← REPLACE
os.environ['QWEN_API_URL'] = ''
print("✅ Credentials set")

In [ ]:
# CELL 6: Quick load model into GPU memory
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"✅ Model loaded on: {model.device}")

In [ ]:
# CELL 7: Define generate_answer function
import json
SYSTEM_PROMPT = """You are an expert electronics assistant.
You are given structured data extracted from a semiconductor datasheet via a Knowledge Graph.
Your job is to generate a clean, concise, and accurate answer based ONLY on the provided data.

STRICT RULES:
1. DO NOT hallucinate or guess missing values
2. DO NOT add any external knowledge
3. DO NOT include unrelated parameters
4. ONLY use the provided structured data
5. If no data is provided, respond: "No structured data found"

OUTPUT FORMAT:
1. Start with the parameter name
2. Provide a short 1-line explanation of what this parameter means
3. List all values clearly with conditions and units
"""

def generate_answer(query: str, data: dict, system_prompt: str = "") -> str:
    if not system_prompt:
        system_prompt = SYSTEM_PROMPT

    if "rows" in data and "columns" in data:
        formatted_rows = [{col: row.get(col, "") for col in data["columns"] if row.get(col)} for row in data["rows"]]
        data_str = json.dumps({"type": "parameter_table", "columns": data["columns"], "records": formatted_rows}, indent=2)
    else:
        data_str = json.dumps(data, indent=2)

    user_message = (
        f"User question: {query}\n\n"
        f"Knowledge Graph Data:\n{data_str}\n\n"
        f"Generate a clean, accurate answer using ONLY the data above. Do not add any information that is not present in the data."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, top_p=0.9, do_sample=True)
    generated = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("✅ generate_answer defined")

In [ ]:
# CELL 8: Patch ai_client for in-process Qwen
ai_client_code = '''
import logging
logger = logging.getLogger(__name__)
_generate_fn = None

def set_generate_function(fn):
    global _generate_fn
    _generate_fn = fn

def is_qwen_available() -> bool:
    return _generate_fn is not None

async def format_with_qwen(query: str, structured_data: dict) -> str:
    if _generate_fn is None:
        logger.warning("Qwen not loaded")
        return ""
    if isinstance(structured_data, dict) and "message" in structured_data:
        if "no " in structured_data["message"].lower():
            return ""
    try:
        return _generate_fn(query, structured_data)
    except Exception as e:
        logger.error("Qwen error: %s", e)
        return ""
'''

with open('backend/app/services/ai_client.py', 'w') as f:
    f.write(ai_client_code)

from backend.app.services.ai_client import set_generate_function, is_qwen_available
set_generate_function(generate_answer)
print(f"✅ Qwen 3.5 4B wired to backend. Available: {is_qwen_available()}")

In [ ]:
# CELL 9: Start ngrok tunnel
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← REPLACE!
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("=" * 60)
print(f"🌐 PUBLIC URL: {public_url}")
print("=" * 60)

In [ ]:
# CELL 10: Start server with nohup
import os
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'
!pkill -f uvicorn 2>/dev/null
!nohup bash -c "cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}:$PYTHONPATH uvicorn backend.main:app --host 0.0.0.0 --port 8000" > {PROJECT_ROOT}/backend.log 2>&1 &
import time; time.sleep(3)
!tail -n 15 {PROJECT_ROOT}/backend.log

In [ ]:
# CELL 11: Health check
import requests
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f"✅ Server UP: {r.json()}")
except Exception as e:
    print(f"❌ Not responding: {e}")